In [ ]:
import pandas as pd
import geopandas as gpd
import osmnx as ox
from shapely.geometry import Polygon, Point
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
from collections import Counter
import os
import datetime as dt

In [ ]:
mvi = gpd.read_file('../data/2_Estructura_via/estructura_via_completa/MVI.geojson')
mvi = mvi.to_crs(epsg = 4326)
mvi["id_mvi"] = mvi["MVICIV"].astype(str) + "_" + mvi["MVISVIA"].str.replace(" ", "")
mvi['contador'] = mvi.groupby('id_mvi').cumcount() + 1
mvi['id_mvi'] = mvi['id_mvi'] + '.' + mvi['contador'].astype(str)
mvi = (
    mvi.explode(ignore_index=True)          # if shapefile has split segments
       .dissolve(by='id_mvi')               # merge all segments of same road
       .reset_index()
)
mvi['road_id'] = mvi['id_mvi']
buffer_distance = 4
mvi_buffered = mvi.to_crs("EPSG:3116").copy()
mvi_buffered['mvi_length'] = mvi_buffered.geometry.length
mvi_buffered['geometry'] = mvi_buffered.geometry.buffer(buffer_distance, cap_style="square")
mvi_buffered_buffer = mvi_buffered.set_geometry('geometry')
mvi_buffered_buffer = mvi_buffered_buffer.to_crs("EPSG:4326")

In [ ]:
road_state_gdf = gpd.read_file('../data/14_Estado_malla_vial/EstadoMallaVialOD_IDU_Sem12024/EstadoMallaVialUrbanaConsolidado.shp').to_crs(epsg = 4326)


In [ ]:
road_state_dict = {'SATISFACTORIO':3,'BUENO':2,'ACEPTABLE':1,'SIN DATO':0,'POBRE':-1,'MUY POBRE':-2,'FALLADO':-3,'GRAVE':-4}
road_state_gdf['numeric_road_state'] = list(tqdm(map(lambda x: road_state_dict[x], road_state_gdf['ESTAASHTO'])))
road_state_gdf['numeric_state_index'] = list(tqdm(map(lambda x: int(x), road_state_gdf['INDEST'])))

In [ ]:
coi_parques = coi_parques.to_crs("EPSG:4326")
coi_parques_intersections = gpd.overlay(coi_parques, mvi_buffered_buffer[['id_mvi', 'mvi_length', 'geometry']], how='intersection')

In [ ]:
road_state_gdf_work = road_state_gdf[['CIV','numeric_road_state','INDEST','LOCEST', 'SUPERFEST', 'KMCAINDEST']]

In [ ]:
roads_gdf[['road_id','MVICIV']]

In [ ]:
road_state_gdf.sort_values(['CIV'])

In [ ]:
pd.merge(roads_gdf[['road_id','MVICIV']], road_state_gdf_work, left_on='MVICIV', right_on='CIV')#.drop(['MVICIV','CIV'],axis=1)

In [ ]:
road_state_gdf_work.corr()

In [ ]:
road_state_gdf.KMCAINDEST.hist(bins=100)

In [ ]:
road_state_gdf.KMCAINDEST.unique()

In [ ]:
road_state_gdf.columns

In [ ]:
road_state_gdf

In [ ]:
{
    'SATISFACTORIO':3,
    'BUENO':2,
    'ACEPTABLE':1,
    'SIN DATO':0,
    'POBRE':-1,
    'MUY POBRE':-2,
    'FALLADO':-3,
    'GRAVE':-4,
}